# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import warnings

# For cleaner output
warnings.filterwarnings('ignore')

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"Dataset: {metadata.name}")
print(f"Description: {metadata.description}")


## 2. Data Overview
Review available record sets and fields using their `@id`s.

In [ ]:
# List all record sets in the dataset
record_sets = list(metadata.record_sets)
print(f"Number of record sets found: {len(record_sets)}")

for rs in record_sets:
    print(f"- RecordSet @id: {rs['@id']} | name: {rs.get('name', '(no name)')}")
    # List fields in this recordset
    if 'field' in rs:
        print("  Fields (@id and name):")
        for fld in rs['field']:
            if isinstance(fld, dict):
                print(f"    - {fld.get('@id')} | name: {fld.get('name')}")
            else:
                print(f"    - {fld}")
    print('')

# Choose a record set to print a sample record
if record_sets:
    example_recordset_id = record_sets[0]['@id']
    print(f"Sample record from record set {example_recordset_id} (by @id):")
    try:
        for i, rec in enumerate(dataset.records(record_set=example_recordset_id)):
            print(json.dumps(rec, indent=2))
            if i >= 0:
                break
    except Exception as e:
        print(f"Failed to read sample record: {e}")


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Collect the @ids of all record sets
record_set_ids = [rs['@id'] for rs in metadata.record_sets]
# Load each record set as a DataFrame
dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {df.shape[0]} records for RecordSet '{record_set_id}'")
    except Exception as e:
        print(f"Failed loading records for {record_set_id}: {e}")

if len(dataframes) > 0:
    main_record_set_id = record_set_ids[0]
    print(f"Fields for record set '{main_record_set_id}':")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# --- Customize these field @id's as needed for your main record set ---
# Try to infer a numeric field from the dataframe
main_df = dataframes[main_record_set_id]

# Print sample field names and types
print('Field names and inferred dtypes:')
print(main_df.dtypes)

# Choose a numeric field (e.g., columns containing 'coverage' or 'MW (kDa)' if present)
try:
    numeric_candidates = [c for c in main_df.columns if pd.api.types.is_numeric_dtype(main_df[c])]
    if numeric_candidates:
        numeric_field = numeric_candidates[0]
    else:
        # Try to cast a likely numeric column (e.g. coverage, MW, etc.)
        for c in main_df.columns:
            try:
                main_df[c] = pd.to_numeric(main_df[c])
                numeric_field = c
                break
            except:
                continue
        else:
            numeric_field = main_df.columns[0]  # fallback
except Exception as e:
    print(f"Failed to choose numeric field: {e}")
    numeric_field = None

# Set a threshold for filtering (arbitrarily 10 unless NaN)
threshold = 10
if numeric_field:
    filtered_df = main_df.copy()
    # Remove NaNs and invalids
    filtered_df = filtered_df[pd.to_numeric(filtered_df[numeric_field], errors='coerce').notnull()]
    filtered_df[numeric_field] = pd.to_numeric(filtered_df[numeric_field])
    filtered_df = filtered_df[filtered_df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by a likely grouping categorical field
    # For example, by 'description' or the first object dtype column
    group_fields = [c for c in filtered_df.columns if pd.api.types.is_object_dtype(filtered_df[c])]
    grouped_df = None
    if group_fields:
        group_field = group_fields[0]
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped data by {group_field} (mean {numeric_field}):")
        print(grouped_df.head())
else:
    print('No suitable numeric field found for EDA.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field and filtered_df.shape[0] > 0:
    plt.figure(figsize=(8,5))
    sns.histplot(filtered_df[numeric_field], kde=True, bins=20)
    plt.title(f'Distribution of {numeric_field} (> {threshold})')
    plt.xlabel(numeric_field)
    plt.show()

    if 'group_field' in locals() and group_field in filtered_df.columns:
        # Barplot group means
        top_groups = grouped_df.head(10).sort_values(numeric_field, ascending=False)
        plt.figure(figsize=(10,5))
        sns.barplot(data=top_groups, x=group_field, y=numeric_field)
        plt.title(f'Top groups by mean {numeric_field}')
        plt.xticks(rotation=45)
        plt.show()
else:
    print('Visualization not possible due to lack of numeric data or filtered records.')

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset provides rich records on proteins isolated from human mast cell extracellular vesicles.
- We demonstrated loading Croissant metadata, identifying record sets and fields by `@id`, and extracting data tables using `mlcroissant`.
- Simple EDA steps (filtering and normalization) and visualizations were performed on the numeric protein abundance field(s).
- Users are encouraged to further analyze, visualize, and model this dataset for their bioinformatics applications.